# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\n{metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# The FAIR^2 dataset contains one main record set.
# We'll inspect available record sets and their fields.

# Fetch all record sets entities
record_sets = dataset.metadata.record_sets
print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name','N/A')}, description: {rs.get('description','N/A')}")

# For each record set, list its fields and their @id
for rs in record_sets:
    print(f"\nFields for record set @id {rs['@id']}:")
    fields = rs.get('field', [])
    for field in fields:
        fname = field.get('name', 'N/A')
        print(f"  - @id: {field['@id']}, name: {fname}, type: {field.get('dataType','N/A')}")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as revealed above.

In [ ]:
# Extract data from each record set
dataframes = {}

# List all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns (@id) for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. 

**Note:** All fields are referenced using their `@id`. Update the chosen numeric field and group field according to the actual field `@id` found above.

In [ ]:
# Choose the main record set to work with (edit as needed)
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Example: Suppose '@id' for numeric field is 'https://api.app.sen.science/frontiers/7862866/field/age'
# and group field is 'https://api.app.sen.science/frontiers/7862866/field/anatomical_location'
numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/field/age'  # Update if needed
group_field_id = 'https://api.app.sen.science/frontiers/7862866/field/anatomical_location'  # Update if needed

if numeric_field_id in df.columns:
    threshold = 45
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by anatomical location field and show means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df)
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset to support analysis.

Here we plot the distribution of age for different anatomical locations, using the record set field `@id`s, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot age distribution by anatomical location if fields exist
if numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title("Age Distribution per Anatomical Location")
    plt.xticks(rotation=30)
    plt.show()
else:
    print("Necessary fields for visualization not found in DataFrame.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides detailed clinical and molecular records for cancer survivors with second primary colorectal cancer.
- All data fields and entities were referenced via their `@id` to ensure schema consistency and reproducibility.
- You can filter, group, and visualize the data with standard Python tools once loaded via `mlcroissant`.
- The schema and metadata provide information for further analysis, including additional record sets and fields.

For more advanced analysis or custom queries, refer to the field and record set `@id`s as reviewed above and consult the [mlcroissant documentation](https://mlcroissant.readthedocs.io/en/latest/).
